<a href="https://colab.research.google.com/github/saifulbutex/journal-quartile-checking/blob/main/Quartile_checking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import os
import re
import pandas as pd
import requests
from google.colab import files

# File paths
scimago_file = "scimago_data.csv"
scimago_excel = "scimago_data.xlsx"

# Helper to load SCImago reference data
def load_scimago_df():
    if os.path.exists(scimago_excel):
        try:
            return pd.read_excel(scimago_excel)
        except Exception:
            pass
    if os.path.exists(scimago_file):
        try:
            return pd.read_csv(scimago_file, sep=";")
        except Exception:
            try:
                return pd.read_csv(scimago_file, sep=";", encoding="latin-1")
            except Exception:
                try:
                    return pd.read_excel(scimago_file)
                except Exception:
                    pass
    return None

# Standardize SCImago columns
def standardize_scimago_columns(df):
    if df is None:
        return None
    col_mapping = {col.strip().lower(): col for col in df.columns}

    title_key = next((col_mapping[k] for k in ['title', 'journal title'] if k in col_mapping), None)
    q_key = next((col_mapping[k] for k in ['sjr best q', 'sjr best quartile', 'quartile'] if k in col_mapping), None)
    issn_key = next((col_mapping[k] for k in ['issn'] if k in col_mapping), None)
    sjr_key = next((col_mapping[k] for k in ['sjr'] if k in col_mapping), None)
    if_key = next((col_mapping[k] for k in ['cites / doc. (2years)', 'cites/doc. (2years)'] if k in col_mapping), None)

    if title_key and q_key:
        rename_dict = {title_key: 'Title', q_key: 'SJR Best Q'}
        if issn_key:
            rename_dict[issn_key] = 'ISSN'
        if sjr_key:
            rename_dict[sjr_key] = 'SJR Score'
        if if_key:
            rename_dict[if_key] = 'Impact Factor Equivalent (2y)'
        return df.rename(columns=rename_dict)
    return None

# --- MEMORY OPTIMIZATION: Skip reloading if already in memory ---
if 'df_scimago' in globals() and df_scimago is not None and 'SJR Score' in df_scimago.columns:
    print("☑️ SCImago reference dataset already loaded in memory. Skipping setup!")
else:
    df_scimago = load_scimago_df()
    if df_scimago is not None:
        standardized_df = standardize_scimago_columns(df_scimago)
        if standardized_df is None:
            print("Warning: The cached reference file is invalid.")
            if os.path.exists(scimago_file): os.remove(scimago_file)
            if os.path.exists(scimago_excel): os.remove(scimago_excel)
            df_scimago = None
        else:
            df_scimago = standardized_df

    if df_scimago is None:
        try:
            print("Attempting to download SCImago dataset...")
            scimago_url = "https://www.scimagojr.com/journalrank.php?out=xls"
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36',
                'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
                'Accept-Language': 'en-US,en;q=0.9'
            }
            response = requests.get(scimago_url, headers=headers, timeout=15)
            response.raise_for_status()

            with open(scimago_file, 'wb') as out_file:
                out_file.write(response.content)
            print("Download complete!\n")
            df_scimago = load_scimago_df()
            if df_scimago is not None:
                df_scimago = standardize_scimago_columns(df_scimago)
        except Exception as e:
            print(f"\nCould not download automatically (Error: {e})")
            print("\n--- ACTION REQUIRED: SCIMAGO REFERENCE DATA ---")
            print("1. Download the SCImago reference file manually here: https://www.scimagojr.com/journalrank.php?out=xls")
            print("2. Upload that reference file below (usually named 'journalrank' with a .csv or .xlsx extension):")

            uploaded_scimago = files.upload()
            if uploaded_scimago:
                uploaded_name = list(uploaded_scimago.keys())[0]
                if uploaded_name.endswith(('.xlsx', '.xls')):
                    if os.path.exists(scimago_excel): os.remove(scimago_excel)
                    os.rename(uploaded_name, scimago_excel)
                else:
                    if os.path.exists(scimago_file): os.remove(scimago_file)
                    os.rename(uploaded_name, scimago_file)
                raw_df = load_scimago_df()
                df_scimago = standardize_scimago_columns(raw_df)
                if df_scimago is not None:
                    print("\nSCImago reference database loaded successfully!\n")
                else:
                    if os.path.exists(scimago_file): os.remove(scimago_file)
                    if os.path.exists(scimago_excel): os.remove(scimago_excel)
                    raise ValueError("Uploaded file does not appear to be a valid SCImago dataset.")
            else:
                raise FileNotFoundError("No SCImago reference dataset provided. Please upload the file to proceed.")
    else:
        print("SCImago reference database is ready.\n")

# Create mappings dynamically to hold Quartile, SJR, and 2-Year IF equivalent
if 'journal_metrics_map' not in globals() or 'issn_metrics_map' not in globals():
    df_scimago["Clean_Title"] = df_scimago["Title"].astype(str).str.strip().str.lower()

    def parse_issns(issn_str):
        if pd.isna(issn_str):
            return []
        issns = str(issn_str).replace("-", "").replace(",", " ").split()
        return [i.strip() for i in issns if i.strip()]

    issn_metrics_map = {}
    journal_metrics_map = {}

    for idx, row in df_scimago.iterrows():
        quartile = str(row["SJR Best Q"]) if pd.notna(row["SJR Best Q"]) else ""
        sjr_score = row.get("SJR Score", "")
        impact_factor = row.get("Impact Factor Equivalent (2y)", "")


        try:
            if pd.notna(sjr_score):
                sjr_score = float(str(sjr_score).replace(",", "."))
        except ValueError:
            pass
        try:
            if pd.notna(impact_factor):
                impact_factor = float(str(impact_factor).replace(",", "."))
        except ValueError:
            pass

        metrics = {"Quartile": quartile, "SJR": sjr_score, "ImpactFactor_2y": impact_factor}

        clean_title = row["Clean_Title"]
        journal_metrics_map[clean_title] = metrics

        if "ISSN" in df_scimago.columns:
            for issn in parse_issns(row["ISSN"]):
                issn_metrics_map[issn] = metrics

def lookup_journal_metrics(query):
    default_metrics = {"Quartile": "", "SJR": "", "ImpactFactor_2y": ""}
    if not query or pd.isna(query):
        return default_metrics

    clean_query = str(query).strip().lower().replace("-", "")
    if clean_query.isdigit() and len(clean_query) == 8:
        if clean_query in issn_metrics_map:
            return issn_metrics_map[clean_query]

    clean_title_query = str(query).strip().lower()
    if clean_title_query in journal_metrics_map:
        return journal_metrics_map[clean_title_query]

    return default_metrics

# Helper function to parse RIS file format
def parse_ris_file(filepath):
    entries = []
    current_entry = {}

    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            lines = f.readlines()
    except UnicodeDecodeError:
        with open(filepath, 'r', encoding='latin-1') as f:
            lines = f.readlines()

    for line in lines:
        line = line.strip()
        if not line:
            continue

        match = re.match(r'^([A-Z0-9]{2})\s*-\s*(.*)$', line)
        if match:
            tag, value = match.groups()
            if tag == 'TY':
                if current_entry:
                    entries.append(current_entry)
                current_entry = {'Type': value}
            elif tag == 'ER':
                if current_entry:
                    entries.append(current_entry)
                    current_entry = {}
            else:
                if current_entry is not None:
                    if tag in current_entry:
                        if isinstance(current_entry[tag], list):
                            current_entry[tag].append(value)
                        else:
                            current_entry[tag] = [current_entry[tag], value]
                    else:
                        current_entry[tag] = value

    if current_entry:
        entries.append(current_entry)

    structured_data = []
    for entry in entries:
        title = entry.get('T1', entry.get('TI', ''))
        if isinstance(title, list): title = " ".join(title)

        year = entry.get('PY', entry.get('Y1', ''))
        if isinstance(year, list): year = year[0]
        year_match = re.search(r'\b\d{4}\b', str(year))
        year = year_match.group(0) if year_match else year

        journal = entry.get('JO', entry.get('JF', entry.get('JA', entry.get('T2', ''))))
        if isinstance(journal, list): journal = journal[0]

        authors = entry.get('AU', '')
        if isinstance(authors, list):
            authors = "; ".join(authors)

        issn = entry.get('SN', '')
        if isinstance(issn, list): issn = issn[0]

        doi = entry.get('DO', entry.get('M3', ''))
        if isinstance(doi, list): doi = doi[0]

        publisher = entry.get('PB', '')
        if isinstance(publisher, list): publisher = "; ".join(publisher)

        structured_data.append({
            'Title': title,
            'Author': authors,
            'Year': year,
            'JournalName': journal,
            'ISSN': issn,
            'DOI': doi,
            'Publisher': publisher
        })

    return pd.DataFrame(structured_data)

# Helper function to parse BibTeX (.bib) file format
def parse_bib_file(filepath):
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read()
    except UnicodeDecodeError:
        with open(filepath, 'r', encoding='latin-1') as f:
            content = f.read()

    # Split into separate entries
    raw_entries = re.findall(r'@\w+\s*\{\s*([^,]+),([\s\S]*?)\n\s*\}(?=\s*@|\s*$)', content)
    structured_data = []

    for key, body in raw_entries:
        # Simple regex-based fields extractor supporting nested braces or quotes
        fields = re.findall(r'(\w+)\s*=\s*([{"‘’“”].*?[}"‘’“”]|[\w\d-]+)', body, re.DOTALL)
        entry_dict = {}
        for field, value in fields:
            clean_field = field.strip().lower()
            # Clean outermost quotes, curly braces, and typical bibTeX formatting artifacts
            clean_value = value.strip().strip('{}\u201c\u201d\u2018\u2019"\'')
            clean_value = re.sub(r'\s+', ' ', clean_value) # normalize spacing
            entry_dict[clean_field] = clean_value

        # Resolve standard keys
        title = entry_dict.get('title', '')
        author = entry_dict.get('author', '').replace(' and ', '; ')
        year = entry_dict.get('year', '')
        journal = entry_dict.get('journal', entry_dict.get('booktitle', ''))
        issn = entry_dict.get('issn', '')
        doi = entry_dict.get('doi', '')
        publisher = entry_dict.get('publisher', '')

        structured_data.append({
            'Title': title,
            'Author': author,
            'Year': year,
            'JournalName': journal,
            'ISSN': issn,
            'DOI': doi,
            'Publisher': publisher
        })

    return pd.DataFrame(structured_data)

# ---------------------------------------------------------
# INTERACTIVE INPUT SECTION
# ---------------------------------------------------------
#@title Choose Input Option { run: "user" }
Input_Option = "Option 1: Single Journal Name or ISSN" #@param ["Option 1: Single Journal Name or ISSN", "Option 2: Upload RIS or BibTeX File"]
Journal_Name_or_ISSN = "0040-5175" #@param {type:"string"}

if Input_Option == "Option 1: Single Journal Name or ISSN":
    metrics = lookup_journal_metrics(Journal_Name_or_ISSN)
    print("=" * 50)
    print(f"Query: {Journal_Name_or_ISSN}")
    if metrics["Quartile"] or metrics["SJR"]:
        print(f"Quartile: {metrics['Quartile']}")
        print(f"SJR Score: {metrics['SJR']}")
        print(f"Impact Factor Equivalent (2y): {metrics['ImpactFactor_2y']}")
    else:
        print("Result: Not found in SCImago dataset")
    print("=" * 50)

else:
    print("Please upload your citations file (.ris or .bib):")
    uploaded = files.upload()

    if uploaded:
        filename = list(uploaded.keys())[0]

        if filename.endswith('.bib'):
            print(f"\nParsing BibTeX (.bib) file: {filename}...")
            df_user = parse_bib_file(filename)
        else:
            print(f"\nParsing RIS (.ris) file: {filename}...")
            df_user = parse_ris_file(filename)

        if not df_user.empty:
            print(f"Extracted {len(df_user)} references. Querying metrics...")

            # Resolve metrics using either JournalName or ISSN
            def fetch_row_metrics(row):
                if pd.notna(row['ISSN']) and str(row['ISSN']).strip():
                    res = lookup_journal_metrics(row['ISSN'])
                    if res["Quartile"]:
                        return res
                return lookup_journal_metrics(row['JournalName'])

            resolved_metrics = df_user.apply(fetch_row_metrics, axis=1)
            df_user["Quartile"] = [m["Quartile"] for m in resolved_metrics]
            df_user["SJR Score"] = [m["SJR"] for m in resolved_metrics]
            df_user["Impact Factor Equivalent (2y)"] = [m["ImpactFactor_2y"] for m in resolved_metrics]

            # Reorder columns so Metadata fields (ISSN, DOI, Publisher) are grouped logically
            cols = list(df_user.columns)
            order_targets = ['ISSN', 'DOI', 'Publisher']
            for target in order_targets:
                if target in cols:
                    cols.remove(target)

            # Insert them after 'JournalName'
            j_idx = cols.index('JournalName') if 'JournalName' in cols else len(cols) - 1
            for i, target in enumerate(order_targets):
                if target in df_user.columns:
                    cols.insert(j_idx + 1 + i, target)

            df_user = df_user[cols]

            output_filename = "Updated_Journals_with_Quartiles.xlsx"
            temp_filename = "temp_raw_output.xlsx"
            df_user.to_excel(temp_filename, index=False)

            # Apply Excel Table formatting
            from openpyxl import load_workbook
            from openpyxl.worksheet.table import Table, TableStyleInfo

            wb = load_workbook(temp_filename)
            ws = wb.active

            max_row = ws.max_row
            max_col = ws.max_column

            from openpyxl.utils import get_column_letter
            ref_range = f"A1:{get_column_letter(max_col)}{max_row}"

            tab = Table(displayName="CitationsTable", ref=ref_range)
            style = TableStyleInfo(name="TableStyleMedium2", showFirstColumn=False,
                                   showLastColumn=False, showRowStripes=True, showColumnStripes=False)
            tab.tableStyleInfo = style
            ws.add_table(tab)

            wb.save(output_filename)
            wb.close()
            if os.path.exists(temp_filename): os.remove(temp_filename)

            print(f"\nSuccessfully processed {len(df_user)} citations!")
            files.download(output_filename)
        else:
            print("Error: Could not extract any valid citation records from the uploaded file.")

☑️ SCImago reference dataset already loaded in memory. Skipping setup!
Query: 0040-5175
Quartile: Q2
SJR Score: 0.498
Impact Factor Equivalent (2y): 
